# Deep Learning Intro -- MNIST Digit Classification with PyTorch

**Dataset:** MNIST (70,000 handwritten digit images, 28x28 pixels, 10 classes)
**Model:** Feedforward neural network -- 784 -> 256 -> 128 -> 10
**Framework:** PyTorch 2.0+

This notebook introduces core PyTorch concepts: tensors, DataLoaders, `nn.Module`, the training loop (forward pass -> loss -> backward -> update), and evaluation. The same pattern scales to CNNs, RNNs, and transformers.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device:    {DEVICE}")

## 1. Load MNIST

We normalize pixel values to the MNIST dataset mean (0.1307) and std (0.3081) to speed up training. A `DataLoader` handles batching and shuffling automatically.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST channel mean and std
])

train_set = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_set  = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_set):,}  |  Test samples: {len(test_set):,}")
print(f"Input shape:   {train_set[0][0].shape}  (channels, H, W)")
print(f"Classes:       digits 0-9")

## 2. Visualize Sample Images

In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, lbl in zip(axes.flatten(), images[:16], labels[:16]):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(lbl.item()), fontsize=10)
    ax.axis('off')
plt.suptitle('Sample MNIST Training Images', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Define the Neural Network

Architecture: **784 -> 256 -> 128 -> 10**

- **Input:** 784 neurons (28x28 flattened)
- **Hidden 1:** 256 neurons + ReLU + Dropout(0.2)
- **Hidden 2:** 128 neurons + ReLU + Dropout(0.2)
- **Output:** 10 neurons (one logit per digit class)

`Dropout` randomly zeros 20% of activations during training to reduce overfitting. It is automatically disabled during `model.eval()`.

In [ ]:
class MNISTNet(nn.Module):
    """Simple feedforward network: 784 -> 256 -> 128 -> 10"""

    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),          # (batch, 1, 28, 28) -> (batch, 784)
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),   # 10 output logits
        )

    def forward(self, x):
        return self.network(x)


model = MNISTNet().to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

## 4. Training Loop

The core PyTorch training pattern (same for any architecture):

1. Zero accumulated gradients (`optimizer.zero_grad()`)
2. Forward pass -- compute predictions
3. Compute loss
4. Backward pass -- compute gradients (`loss.backward()`)
5. Update weights (`optimizer.step()`)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()     # enables Dropout
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()            # reset gradients
        outputs = model(images)          # forward pass
        loss = criterion(outputs, labels)
        loss.backward()                  # compute gradients
        optimizer.step()                 # update weights
        total_loss += loss.item() * images.size(0)
        correct    += outputs.argmax(1).eq(labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()      # disables Dropout
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():   # no gradient computation needed
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * images.size(0)
            correct    += outputs.argmax(1).eq(labels).sum().item()
            total      += images.size(0)
    return total_loss / total, correct / total


print("Training and evaluation functions defined.")

In [ ]:
EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    vl_loss, vl_acc = evaluate(model,    test_loader,  criterion, DEVICE)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f"Epoch {epoch:2d} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.1f}% | "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc*100:.1f}%")

## 5. Training Curves

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(epochs, history['train_loss'], 'o-', label='Train',      color='#3498db')
axes[0].plot(epochs, history['val_loss'],   'o-', label='Validation', color='#e74c3c')
axes[0].set_title('Loss Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()

axes[1].plot(epochs, [a * 100 for a in history['train_acc']], 'o-',
             label='Train', color='#3498db')
axes[1].plot(epochs, [a * 100 for a in history['val_acc']],   'o-',
             label='Validation', color='#e74c3c')
axes[1].set_title('Accuracy Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Final Test Accuracy: {history['val_acc'][-1] * 100:.2f}%")

## 6. Visualize Predictions

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images.to(DEVICE)).argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, true, pred in zip(axes.flatten(), images[:16], labels[:16], preds[:16]):
    ax.imshow(img.squeeze(), cmap='gray')
    color = '#2ecc71' if true == pred else '#e74c3c'
    ax.set_title(f'T:{true} P:{pred}', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Test Predictions (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()

## Results and Next Steps

A simple 3-layer feedforward network reaches **~97-98% accuracy** on MNIST in 10 epochs.

**What this notebook demonstrated:**
- PyTorch tensor operations and GPU-agnostic code via `DEVICE`
- `nn.Module` for defining reusable model architectures
- `DataLoader` for batched, shuffled data iteration
- The standard train/eval loop with `model.train()` and `model.eval()`
- `torch.no_grad()` to disable gradients during evaluation

**Natural next steps:**
- Replace `nn.Flatten + Linear` layers with convolutional layers (`nn.Conv2d`) for better image understanding
- Add a learning rate scheduler (`torch.optim.lr_scheduler`)
- Apply the same training loop to a tabular healthcare classification problem